# Layer 4: Orderbook Dynamics


In [ ]:
import os
import numpy as np
import pandas as pd

from mda.loader import load_orderbook, load_orderbook_updates
from mda.timestamps import add_ts_columns
from mda import EXCHANGES, DATA_DIR
from mda.layer4.bbo import compute_bbo, spread_stats
from mda.layer4.update_rates import (
    compute_update_rate_by_depth,
    compute_delta_compression_ratio,
)
from mda.layer4.level_lifetime import compute_level_lifetime
from mda.plots.charts import (
    save_figure,
    plot_bbo_spread_depth,
    plot_update_rate_by_depth,
    plot_delta_compression,
    plot_orderbook_depth_profile,
    plot_level_lifetime,
)


In [ ]:
DATA_DIR = "/data/parquet"
REPORTS_DIR = "/data/reports"
os.makedirs(REPORTS_DIR, exist_ok=True)

# Load BBO only first to check memory footprint
ob = load_orderbook(DATA_DIR, exchanges=EXCHANGES, bbo_only=True)
print("BBO shape:", ob.shape)
print(f"Memory usage: {ob.memory_usage(deep=True).sum() / 1e6:.1f} MB")


In [ ]:
bbo = compute_bbo(ob)
s = spread_stats(bbo)

print("Spread statistics (bps):")
print(s)


In [ ]:
# Plot O2: BBO spread and depth
fig = plot_bbo_spread_depth(bbo)
fig.show()
save_figure(fig, "O2_bbo_spread_depth", REPORTS_DIR)


In [ ]:
# Load full orderbook up to level 20; iterate per exchange to manage memory
ob_full_parts = []
for exch in EXCHANGES:
    part = load_orderbook(DATA_DIR, exchanges=[exch], bbo_only=False, max_level=20)
    ob_full_parts.append(part)
    print(f"{exch}: {part.shape}")

ob_full = pd.concat(ob_full_parts, ignore_index=True)
print("\nFull orderbook shape:", ob_full.shape)


In [ ]:
update_rate = compute_update_rate_by_depth(ob_full)

print("Update rate by depth level:")
print(update_rate.head(20))


In [ ]:
# Plot O1: Update rate by depth
fig = plot_update_rate_by_depth(update_rate)
fig.show()
save_figure(fig, "O1_update_rate_by_depth", REPORTS_DIR)


In [ ]:
# Load orderbook updates (incremental deltas)
ob_updates = load_orderbook_updates(DATA_DIR, exchanges=EXCHANGES)
print("Orderbook updates shape:", ob_updates.shape)


In [ ]:
# Plot O3: Delta compression ratio
delta = compute_delta_compression_ratio(ob_full, ob_updates)

fig = plot_delta_compression(delta)
fig.show()
save_figure(fig, "O3_delta_compression", REPORTS_DIR)


In [ ]:
# Plot O4: Orderbook depth profile per exchange
for exch in EXCHANGES:
    ob_ex = ob_full[ob_full["exchange"] == exch]
    if ob_ex.empty:
        continue
    fig = plot_orderbook_depth_profile(ob_ex, exchange=exch)
    fig.show()
    save_figure(fig, f"O4_depth_profile_{exch}", REPORTS_DIR)


In [ ]:
# Plot O5: Level lifetime for BTC on each exchange (via Polars for performance)
import polars as pl

for exch in EXCHANGES:
    lifetime_df = compute_level_lifetime(DATA_DIR, exchange=exch, symbol="BTC")
    if lifetime_df is None or (hasattr(lifetime_df, "is_empty") and lifetime_df.is_empty()):
        print(f"No level lifetime data for BTC on {exch}, skipping.")
        continue
    fig = plot_level_lifetime(lifetime_df, exchange=exch)
    fig.show()
    save_figure(fig, f"O5_level_lifetime_BTC_{exch}", REPORTS_DIR)
